# Bayesian optimization of a monopole length

We have a 1-port simulator (here mocked with a closed-form approximation) that maps monopole length `L` to a target metric. Run `yee.bo_minimize` to find the optimum across `L ∈ [0.05, 0.4] m`.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from yee import BoConfig, bo_minimize

Synthetic objective: minimize VSWR at 1 GHz for a quarter-wave monopole. The optimum sits at L = λ/4 ≈ 0.075 m.

In [ ]:
C_LIGHT = 2.99792458e8
F0 = 1.0e9  # 1 GHz

def vswr_objective(x):
    """Closed-form VSWR vs length for a thin-monopole model.
    Minimum at L = λ/4. Quadratic penalty in length offset; sinusoidal phase
    roughness mimics the real surface's non-convex local structure."""
    length_m = float(x[0])
    lam_quarter = C_LIGHT / F0 / 4.0
    offset = (length_m - lam_quarter) / lam_quarter
    # Smooth bowl + small oscillation to give BO a non-trivial surface
    return 1.0 + 50.0 * offset**2 + 0.3 * math.sin(20.0 * length_m)

Run BO with 5 LHS initial samples + 25 EI iterations = 30 total evaluations.

In [ ]:
cfg = BoConfig(n_initial=5, n_iters=25, n_candidates=2048, xi=0.01, seed=42)
result = bo_minimize(vswr_objective, [(0.05, 0.4)], cfg)

print(f"Best length:  {result.x_best[0]*1000:.3f} mm")
print(f"Best VSWR:    {result.y_best:.4f}")
print(f"Closed-form optimum:  {C_LIGHT / F0 / 4.0 * 1000:.3f} mm")
print(f"Total evaluations:     {len(result.history_y)}")

Convergence: best-so-far VSWR over evaluation count.

In [ ]:
ys = np.asarray(result.history_y)
best_so_far = np.minimum.accumulate(ys)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(np.arange(1, len(ys) + 1), best_so_far, marker="o", label="best so far")
ax.scatter(np.arange(1, len(ys) + 1), ys, marker=".", alpha=0.4, label="each eval")
ax.set_xlabel("Evaluation #")
ax.set_ylabel("VSWR")
ax.set_title("BO convergence on synthetic monopole")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Posterior over the design space: BO eval history vs the true objective.

In [ ]:
xs_dense = np.linspace(0.05, 0.4, 400)
ys_dense = np.array([vswr_objective([x]) for x in xs_dense])
xs_eval = result.history_x[:, 0]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(xs_dense * 1000, ys_dense, label="objective (dense)", color="C0")
ax.scatter(xs_eval * 1000, ys, c=np.arange(len(ys)), cmap="viridis",
           label="BO evaluations", zorder=5)
ax.axvline(C_LIGHT / F0 / 4.0 * 1000, ls="--", color="k", label="L = λ/4")
ax.set_xlabel("Length (mm)")
ax.set_ylabel("VSWR")
ax.set_title("Objective + evaluation history")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Next steps — extending this notebook to a real solver

Swap `vswr_objective` for a closure that calls `yee.FdtdDriver` (or `yee.PlanarMoM`) on a parameterized geometry and extracts S₁₁ → VSWR at the design frequency. Future work: multi-objective BO over bandwidth and gain.

See `crates/yee-py/README.md` for the full `bo_minimize` / `FdtdDriver` API surface.